# Dataset Construction

We construct a customer-balanced sample of approximately 2 million merged pull requests created between April 1 and June 30, 2026. 

Eligible customers must have at least 1,000 Copilot seats. We sample pull requests from more than 100 customers while limiting the contribution of any individual customer. 

Customers receive a base allocation, followed by an additional allocation based on the square root of their available PR volume, with a maximum customer-level quota. 

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
import math

sys.path.insert(0, "/Users/syd-kiwi/Code/")

from ecm_mlflow_libs.minioctopy import Trino
from sqlalchemy import create_engine

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

sns.set_theme(style='whitegrid', context='notebook', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

Trino('syd-kiwi').engine.auth
t = Trino('syd-kiwi')

Connecting to Trino as user syd-kiwi...
Connecting to Trino as user syd-kiwi...


In [10]:
import pandas as pd
from sqlalchemy import create_engine

# Trino connection settings
user = t.engine.user
host = t.engine.host
port = t.engine.port
catalog = "delta"
schema = "data_science"
http_scheme = t.engine.http_scheme 
auth = t.engine.auth

con_str = f"trino://{user}@{host}:{port}/{catalog}/{schema}?protocol={http_scheme}"

engine = create_engine(
    con_str,
    connect_args={
        "host": t.engine.host,
        "port": t.engine.port,
        "user": t.engine.user,
        "catalog": "delta",
        "schema": "data_science",
        "http_scheme": t.engine.http_scheme,
        "auth": t.engine.auth,
    }
)

code_complexity_query = f"""
WITH complexity_base AS (
    SELECT
        billable_owner_id AS customer_id,
        user_id,
        repository_id,
        pull_request_id,
        oid,

        CAST(merge_day AS DATE) AS merge_day,
        CAST(DATE_TRUNC('month', CAST(merge_day AS DATE)) AS DATE) AS merge_month,

        had_copilot_seat,

        files_added,
        files_deleted,
        files_modified,
        files_touched,

        lines_modified,
        code_lines_modified,
        config_lines_modified,

        minutes_to_merge,
        num_reviewers,
        num_review_comments,

        is_code_pr,
        standardized_complexity_score,
        complexity_score,

        lines_added_by_language,
        lines_deleted_by_language,
        languages_list

    FROM delta.data_science.code_complexity_current

    WHERE CAST(merge_day AS DATE) >= DATE '2026-01-01'
      AND billable_owner_id = 13213
      AND repository_id IS NOT NULL
      AND pull_request_id IS NOT NULL
),

pr_base AS (
    SELECT
        c.customer_id,
        c.user_id,
        c.repository_id,
        c.pull_request_id,
        c.oid,

        p.title,
        p.body,

        CONCAT(
            COALESCE(p.title, ''),
            ' ',
            COALESCE(p.body, '')
        ) AS combined_text_nlp,

        c.merge_day,
        c.merge_month,
        c.had_copilot_seat,

        c.files_added,
        c.files_deleted,
        c.files_modified,
        c.files_touched,

        c.lines_modified,
        c.code_lines_modified,
        c.config_lines_modified,

        c.minutes_to_merge,
        c.num_reviewers,
        c.num_review_comments,

        c.is_code_pr,
        c.standardized_complexity_score,
        c.complexity_score,

        c.lines_added_by_language,
        c.lines_deleted_by_language,
        c.languages_list

    FROM complexity_base c

    LEFT JOIN delta.canonical.pull_requests_current p
        ON c.pull_request_id = p.id
       AND c.repository_id = p.repository_id
)

SELECT *
FROM pr_base
ORDER BY merge_day, repository_id, pull_request_id
"""

from pathlib import Path
from sqlalchemy import text

output_dir = Path("/Users/syd-kiwi/Code/pr-complexity-analysis/data/microsoft_customer_parquet")
output_dir.mkdir(parents=True, exist_ok=True)

# Remove previous chunk files
for old_file in output_dir.glob("part--*.parquet"):
    old_file.unlink()

total_rows = 0
total_parts = 0
chunk_size = 5_000

with engine.connect() as connection:

    stream_connection = connection.execution_options(
        stream_results=True,
        max_row_buffer=chunk_size
    )

    for part_number, chunk_df in enumerate(
        pd.read_sql_query(
            text(code_complexity_query),
            con=stream_connection,
            chunksize=chunk_size
        )
    ):
        output_file = (
            output_dir /
            f"part--{part_number:05d}.parquet"
        )

        chunk_df.to_parquet(
            output_file,
            index=False,
            engine="pyarrow",
            compression="snappy"
        )

        total_rows += len(chunk_df)
        total_parts += 1

        print(
            f"Saved {output_file.name}: "
            f"{len(chunk_df):,} rows | "
            f"Total: {total_rows:,}"
        )

print("\nFinished")
print(f"Total rows saved: {total_rows:,}")
print(f"Total files saved: {total_parts:,}")
print(f"Location: {output_dir}")

Saved part--00000.parquet: 5,000 rows | Total: 5,000
Saved part--00001.parquet: 5,000 rows | Total: 10,000
Saved part--00002.parquet: 5,000 rows | Total: 15,000
Saved part--00003.parquet: 5,000 rows | Total: 20,000
Saved part--00004.parquet: 5,000 rows | Total: 25,000
Saved part--00005.parquet: 5,000 rows | Total: 30,000
Saved part--00006.parquet: 5,000 rows | Total: 35,000
Saved part--00007.parquet: 5,000 rows | Total: 40,000
Saved part--00008.parquet: 5,000 rows | Total: 45,000
Saved part--00009.parquet: 5,000 rows | Total: 50,000
Saved part--00010.parquet: 5,000 rows | Total: 55,000
Saved part--00011.parquet: 5,000 rows | Total: 60,000
Saved part--00012.parquet: 5,000 rows | Total: 65,000
Saved part--00013.parquet: 5,000 rows | Total: 70,000
Saved part--00014.parquet: 5,000 rows | Total: 75,000
Saved part--00015.parquet: 5,000 rows | Total: 80,000
Saved part--00016.parquet: 5,000 rows | Total: 85,000
Saved part--00017.parquet: 5,000 rows | Total: 90,000
Saved part--00018.parquet: 5,